# ENARES 2024 CRS04 — Stage 03
## NB01 · Setup, prerequisitos, llave, colisiones y cleaned
Cubre Issues #22–#28.

In [ ]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes openpyxl XlsxWriter tabulate
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd, hashlib, os
auth.authenticate_user(); drive.mount('/content/drive')
PROJECT_ID='enares-2024-crs04'; LOCATION='US'; EXPECTED_ROWS=18807
ROOT_DRIVE=Path('/content/drive/MyDrive/ENARES_2024_PROJECT')
LOG_DIR=ROOT_DRIVE/'05Resultados'/'logs'/'stage03'; SQL_DIR=ROOT_DRIVE/'02SQL'; OUTPUT_DIR=ROOT_DRIVE/'04Outputs'; DOCS_DIR=ROOT_DRIVE/'docs'; R_DIR=ROOT_DRIVE/'03Scripts_R'
for d in [LOG_DIR,SQL_DIR,OUTPUT_DIR,DOCS_DIR,R_DIR]: d.mkdir(parents=True,exist_ok=True)
RUN_UTC=datetime.now(timezone.utc).isoformat(); client=bigquery.Client(project=PROJECT_ID,location=LOCATION)
display(client.query('SELECT CURRENT_DATE() AS fecha_actual').result().to_dataframe())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.6 MB/s eta 0:00:00
Mounted at /content/drive


,fecha_actual
0,2026-08-11


## 1. Datasets Stage 03

In [ ]:
rows=[]
for name in ['enares2024_crs04_cleaned','enares2024_crs04_analytical','enares2024_crs04_outputs']:
 d=bigquery.Dataset(f'{PROJECT_ID}.{name}'); d.location=LOCATION; x=client.create_dataset(d,exists_ok=True); rows.append({'dataset':name,'location':x.location,'verified_utc':RUN_UTC})
pd.DataFrame(rows).to_csv(LOG_DIR/'stage3_dataset_registry.csv',index=False); display(pd.DataFrame(rows))

,dataset,location,verified_utc
0,enares2024_crs04_cleaned,US,2026-08-11T22:58:50.711147+00:00
1,enares2024_crs04_analytical,US,2026-08-11T22:58:50.711147+00:00
2,enares2024_crs04_outputs,US,2026-08-11T22:58:50.711147+00:00


## 2. Prerequisito Stage 02

In [ ]:
required=['raw_crs04_cap100','raw_crs04_cap200','raw_crs04_cap248','raw_crs04_cap300','metadata_crs04_variables','metadata_crs04_value_labels','metadata_crs04_missing_codes','metadata_crs04_source_files']
rows=[]
for name in required:
 try:
  t=client.get_table(f'{PROJECT_ID}.enares2024_crs04_raw.{name}'); rows.append({'table':name,'exists':True,'rows':t.num_rows,'columns':len(t.schema),'error':None})
 except Exception as e: rows.append({'table':name,'exists':False,'rows':None,'columns':None,'error':str(e)})
stage2=pd.DataFrame(rows); stage2.to_csv(LOG_DIR/'stage3_stage2_prerequisite_check.csv',index=False); display(stage2)
if not stage2['exists'].all(): raise RuntimeError('Stage 02 incompleto')

,table,exists,rows,columns,error
0,raw_crs04_cap100,True,18807,147,None
1,raw_crs04_cap200,True,18807,523,None
2,raw_crs04_cap248,True,18807,578,None
3,raw_crs04_cap300,True,18807,51,None
4,metadata_crs04_variables,True,1299,6,None
5,metadata_crs04_value_labels,True,3317,7,None
6,metadata_crs04_missing_codes,True,0,9,None
7,metadata_crs04_source_files,True,4,29,None


## 3. Llave ID + COLEGIAL_ID

In [ ]:
raw_tables = [
    "raw_crs04_cap100",
    "raw_crs04_cap200",
    "raw_crs04_cap248",
    "raw_crs04_cap300",
]

key_parts = []

for table_name in raw_tables:
    sql = f"""
    WITH key_counts AS (
      SELECT
        ID,
        COLEGIAL_ID,
        COUNT(*) AS key_n
      FROM `{PROJECT_ID}.enares2024_crs04_raw.{table_name}`
      GROUP BY ID, COLEGIAL_ID
    )
    SELECT
      '{table_name}' AS table_name,

      (
        SELECT COUNT(*)
        FROM `{PROJECT_ID}.enares2024_crs04_raw.{table_name}`
      ) AS total_rows,

      (
        SELECT COUNTIF(ID IS NULL)
        FROM `{PROJECT_ID}.enares2024_crs04_raw.{table_name}`
      ) AS id_nulls,

      (
        SELECT COUNTIF(COLEGIAL_ID IS NULL)
        FROM `{PROJECT_ID}.enares2024_crs04_raw.{table_name}`
      ) AS colegial_id_nulls,

      COUNT(*) AS distinct_keys,

      COALESCE(SUM(key_n - 1), 0) AS duplicated_key_rows

    FROM key_counts
    """

    result = client.query(sql).result().to_dataframe()
    key_parts.append(result)

key = pd.concat(key_parts, ignore_index=True)

key.to_csv(
    LOG_DIR / "stage3_key_validation.csv",
    index=False
)

display(key)

failed = (
    (key["id_nulls"] > 0)
    | (key["colegial_id_nulls"] > 0)
    | (key["duplicated_key_rows"] > 0)
    | (key["total_rows"] != EXPECTED_ROWS)
    | (key["distinct_keys"] != EXPECTED_ROWS)
)

if failed.any():
    raise RuntimeError(
        "Llave no aprobada. Revisar stage3_key_validation.csv"
    )

print("Llave ID + COLEGIAL_ID aprobada en las cuatro tablas.")

,table_name,total_rows,id_nulls,colegial_id_nulls,distinct_keys,duplicated_key_rows
0,raw_crs04_cap100,18807,0,0,18807,0
1,raw_crs04_cap200,18807,0,0,18807,0
2,raw_crs04_cap248,18807,0,0,18807,0
3,raw_crs04_cap300,18807,0,0,18807,0


Llave ID + COLEGIAL_ID aprobada en las cuatro tablas.


## 4. Control FLOAT64

In [ ]:
parts=[]
for tname in raw_tables:
 t=client.get_table(f'{PROJECT_ID}.enares2024_crs04_raw.{tname}'); sm={f.name:f.field_type for f in t.schema}
 if sm.get('ID') in ('FLOAT','FLOAT64') or sm.get('COLEGIAL_ID') in ('FLOAT','FLOAT64'):
  sql="SELECT '"+tname+"' table_name, COUNTIF(ID IS NOT NULL AND ID!=FLOOR(ID)) id_with_decimals, COUNTIF(COLEGIAL_ID IS NOT NULL AND COLEGIAL_ID!=FLOOR(COLEGIAL_ID)) colegial_id_with_decimals FROM `"+PROJECT_ID+".enares2024_crs04_raw."+tname+"`"
  parts.append(client.query(sql).result().to_dataframe())
fc=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=['table_name','id_with_decimals','colegial_id_with_decimals']); fc.to_csv(LOG_DIR/'stage3_key_float_validation.csv',index=False); display(fc)
if len(fc) and (fc[['id_with_decimals','colegial_id_with_decimals']]>0).any().any(): raise RuntimeError('Llave con decimales reales')

,table_name,id_with_decimals,colegial_id_with_decimals
0,raw_crs04_cap100,0,0
1,raw_crs04_cap200,0,0
2,raw_crs04_cap248,0,0
3,raw_crs04_cap300,0,0


## 5. Contrato de variables

In [ ]:
required_vars = [
    "ID",
    "COLEGIAL_ID",
    "FACTOR_ALUMNOS",
    "CCDD",
    "C3ANIO",
    "TURNO",
    "C3SECC",
    "SEXO",
    "AREA",
    "C3P128",
    "C4P129",
    "C3P203",
    "C3P207",
    "C3P225",
    "C3P229",
]

required_vars += [f"C3P201_{i}" for i in range(1, 12)]
required_vars += [f"C3P205_{i}" for i in range(1, 8)]
required_vars += [f"C3P223_{i}" for i in range(1, 15)]
required_vars += [f"C3P227_{i}" for i in range(1, 11)]
required_vars += [f"C4P248_{i}" for i in range(1, 17)]

required_vars = sorted(set(required_vars))
sql='SELECT DISTINCT CAST(variable_name AS STRING) variable_name FROM `'+PROJECT_ID+'.enares2024_crs04_raw.metadata_crs04_variables`'
meta=client.query(sql).result().to_dataframe()['variable_name'].tolist(); contract=pd.DataFrame({'variable':required_vars}); contract['present_in_metadata']=contract['variable'].isin(meta); contract.to_csv(LOG_DIR/'stage3_variable_contract_check.csv',index=False); display(contract)
missing_vars = contract.loc[
    ~contract["present_in_metadata"],
    "variable"
].tolist()

if missing_vars:
    raise RuntimeError(
        "Contrato no aprobado. Variables ausentes: "
        + ", ".join(missing_vars)
    )

,variable,present_in_metadata
0,AREA,True
1,C3ANIO,True
2,C3P128,True
3,C3P201_1,True
4,C3P201_10,True
...,...,...
68,COLEGIAL_ID,True
69,FACTOR_ALUMNOS,True
70,ID,True
71,SEXO,True


## 6. Colisiones

In [ ]:
collision_sql = f"""
SELECT
  column_name,
  COUNT(DISTINCT table_name) AS n_tables,
  STRING_AGG(DISTINCT table_name ORDER BY table_name) AS tables
FROM `{PROJECT_ID}.enares2024_crs04_raw.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN (
  'raw_crs04_cap100',
  'raw_crs04_cap200',
  'raw_crs04_cap248',
  'raw_crs04_cap300'
)
GROUP BY column_name
HAVING n_tables > 1
ORDER BY n_tables DESC, column_name
"""

coll = client.query(collision_sql).result().to_dataframe()
coll.to_csv(
    LOG_DIR / "stage3_column_collisions.csv",
    index=False
)
display(coll)

,column_name,n_tables,tables
0,AREA,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."
1,C3ANIO,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."
2,C3SECC,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."
3,CCDD,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."
4,CCDI,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."
5,CCPP,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."
6,CODCCPP,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."
7,COLEGIAL_ID,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."
8,DEPARTAMENTO,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."
9,DIREED,4,"raw_crs04_cap100,raw_crs04_cap200,raw_crs04_ca..."


## 7. Resolver colisiones explícitamente

In [ ]:
def get_columns(table_name):
    table = client.get_table(
        f"{PROJECT_ID}.enares2024_crs04_raw.{table_name}"
    )
    return [field.name for field in table.schema]


cap100_cols = set(get_columns("raw_crs04_cap100"))
cap200_cols = set(get_columns("raw_crs04_cap200"))
cap248_cols = set(get_columns("raw_crs04_cap248"))
cap300_cols = set(get_columns("raw_crs04_cap300"))

EXCLUDE_FROM_CAP200 = sorted(cap100_cols & cap200_cols)

existing_after_cap200 = cap100_cols | cap200_cols
EXCLUDE_FROM_CAP248 = sorted(existing_after_cap200 & cap248_cols)

existing_after_cap248 = existing_after_cap200 | cap248_cols
EXCLUDE_FROM_CAP300 = sorted(existing_after_cap248 & cap300_cols)

print("CAP200 excluidas:", len(EXCLUDE_FROM_CAP200))
print("CAP248 excluidas:", len(EXCLUDE_FROM_CAP248))
print("CAP300 excluidas:", len(EXCLUDE_FROM_CAP300))
collision_resolution = pd.DataFrame(
    [
        {
            "secondary_table": "raw_crs04_cap200",
            "excluded_columns": ";".join(EXCLUDE_FROM_CAP200),
        },
        {
            "secondary_table": "raw_crs04_cap248",
            "excluded_columns": ";".join(EXCLUDE_FROM_CAP248),
        },
        {
            "secondary_table": "raw_crs04_cap300",
            "excluded_columns": ";".join(EXCLUDE_FROM_CAP300),
        },
    ]
)

collision_resolution.to_csv(
    LOG_DIR / "stage3_column_collision_resolution.csv",
    index=False
)

display(collision_resolution)

CAP200 excluidas: 31
CAP248 excluidas: 31
CAP300 excluidas: 31


,secondary_table,excluded_columns
0,raw_crs04_cap200,AREA;C3ANIO;C3SECC;CCDD;CCDI;CCPP;CODCCPP;COLE...
1,raw_crs04_cap248,AREA;C3ANIO;C3SECC;CCDD;CCDI;CCPP;CODCCPP;COLE...
2,raw_crs04_cap300,AREA;C3ANIO;C3SECC;CCDD;CCDI;CCPP;CODCCPP;COLE...


In [ ]:
def exc(columns):
    if not columns:
        return ""
    quoted = ", ".join(f"`{column}`" for column in columns)
    return f" EXCEPT({quoted})"

## 8. Crear cleaned

In [ ]:
cleaned_sql='CREATE OR REPLACE TABLE `'+PROJECT_ID+'.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents` AS SELECT a.*, b.*'+exc(EXCLUDE_FROM_CAP200)+', c.*'+exc(EXCLUDE_FROM_CAP248)+', d.*'+exc(EXCLUDE_FROM_CAP300)+' FROM `'+PROJECT_ID+'.enares2024_crs04_raw.raw_crs04_cap100` a LEFT JOIN `'+PROJECT_ID+'.enares2024_crs04_raw.raw_crs04_cap200` b USING(ID,COLEGIAL_ID) LEFT JOIN `'+PROJECT_ID+'.enares2024_crs04_raw.raw_crs04_cap248` c USING(ID,COLEGIAL_ID) LEFT JOIN `'+PROJECT_ID+'.enares2024_crs04_raw.raw_crs04_cap300` d USING(ID,COLEGIAL_ID)'
(SQL_DIR/'stage3_create_crs04_cleaned.sql').write_text(cleaned_sql,encoding='utf-8'); client.query(cleaned_sql).result(); print('cleaned creada')

cleaned creada


## 9. Validar cleaned y calidad

In [ ]:
sql = f"""
WITH key_counts AS (
  SELECT
    ID,
    COLEGIAL_ID,
    COUNT(*) AS key_n
  FROM `{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
  GROUP BY ID, COLEGIAL_ID
)
SELECT
  (
    SELECT COUNT(*)
    FROM `{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
  ) AS rows_cleaned,

  COUNT(*) AS distinct_keys,

  (
    SELECT COUNTIF(ID IS NULL)
    FROM `{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
  ) AS id_nulls,

  (
    SELECT COUNTIF(COLEGIAL_ID IS NULL)
    FROM `{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
  ) AS colegial_id_nulls,

  COALESCE(SUM(key_n - 1), 0) AS duplicated_key_rows
FROM key_counts
"""

chk = client.query(sql).result().to_dataframe()

chk.to_csv(
    LOG_DIR / "stage3_cleaned_validation.csv",
    index=False
)

display(chk)

r = chk.iloc[0]

if (
    r["rows_cleaned"] != EXPECTED_ROWS
    or r["distinct_keys"] != EXPECTED_ROWS
    or r["id_nulls"] > 0
    or r["colegial_id_nulls"] > 0
    or r["duplicated_key_rows"] > 0
):
    raise RuntimeError("Cleaned inválida")
if r['rows_cleaned']!=EXPECTED_ROWS or r['distinct_keys']!=EXPECTED_ROWS or r['id_nulls']>0 or r['colegial_id_nulls']>0: raise RuntimeError('Cleaned inválida')
parts=[]
for v in ['ID','COLEGIAL_ID','FACTOR_ALUMNOS','SEXO','AREA','CCDD']:
 q='SELECT \''+v+'\' variable, COUNT(*) total_rows, COUNTIF(`'+v+'` IS NULL) null_rows, SAFE_DIVIDE(COUNTIF(`'+v+'` IS NULL),COUNT(*)) null_rate FROM `'+PROJECT_ID+'.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`'; parts.append(client.query(q).result().to_dataframe())
quality=pd.concat(parts,ignore_index=True); quality.to_csv(OUTPUT_DIR/'data_quality_report.csv',index=False); display(quality)

,rows_cleaned,distinct_keys,id_nulls,colegial_id_nulls,duplicated_key_rows
0,18807,18807,0,0,0


,variable,total_rows,null_rows,null_rate
0,ID,18807,0,0.0
1,COLEGIAL_ID,18807,0,0.0
2,FACTOR_ALUMNOS,18807,0,0.0
3,SEXO,18807,0,0.0
4,AREA,18807,0,0.0
5,CCDD,18807,0,0.0


In [ ]:
critical_nulls = quality[
    quality["variable"].isin(
        ["ID", "COLEGIAL_ID", "FACTOR_ALUMNOS", "SEXO", "AREA", "CCDD"]
    )
    & (quality["null_rows"] > 0)
]

if len(critical_nulls) > 0:
    display(critical_nulls)
    raise RuntimeError(
        "Existen nulos en variables críticas. Documentar o corregir antes de continuar."
    )

## 10. Decisión

In [ ]:
with open(
    LOG_DIR / "cleaning_decisions_log.md",
    "a",
    encoding="utf-8"
) as f:
    f.write(
        f"\n\n# NB01 {RUN_UTC}\n"
        "- Llave validada: ID + COLEGIAL_ID.\n"
        "- CAP100 utilizada como tabla ancla.\n"
        "- Prioridad de columnas repetidas: CAP100 > CAP200 > CAP248 > CAP300.\n"
        "- Columnas excluidas registradas en stage3_column_collision_resolution.csv.\n"
        "- Universo esperado y validado: 18,807 adolescentes.\n"
    )
    print('NB01 completado')

NB01 completado
